# 长期记忆-基础API
## 架构
- 长期记忆依赖Store+namespace+key+value
- Store: 支持内存和持久化
- namespace: 命令空间，类型元组
- key: 类型str
- value: 类型dict

## put() / get()
- put(namespace, key, value)
    - 当前版本，对于InMemoryStore，每次put都会新建（即便namespace和key一样），所以最终item的created_at和updated_at都始终相同，而PostgresStore如果namespace和key一样，会更新
- get(namespace, key)


### 基于内存实现 - InMemoryStore

In [4]:
import time

from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

namespace = ("user", "info")
key = "user-1"
value = {
    "name": "Peter"
}
store.put(namespace, key, value)

item = store.get(namespace, key)
print(item)

value2 = {
    "name": "Mary"
}
store.put(namespace, key, value2)
item2 = store.get(namespace, key)
print(item2)

Item(namespace=['user', 'info'], key='user-1', value={'name': 'Peter'}, created_at='2026-07-24T07:08:45.413959+00:00', updated_at='2026-07-24T07:08:45.413960+00:00')
Item(namespace=['user', 'info'], key='user-1', value={'name': 'Mary'}, created_at='2026-07-24T07:08:45.414593+00:00', updated_at='2026-07-24T07:08:45.414595+00:00')


### 基于Postgres实现 - PostgresStore

In [2]:
from langgraph.store.postgres import PostgresStore
import time

from common import load_postgresql_url, init_dashscope_embedding_model

with PostgresStore.from_conn_string(load_postgresql_url()) as store:
    store.setup()

    namespace = ("user", "info")
    key = "user-1"
    value = {
        "name": "Peter"
    }
    store.put(namespace, key, value)

    item = store.get(namespace, key)
    print(item)
    time.sleep(1)
    value = {
        "name": "Ruby"
    }
    store.put(namespace, key, value)
    item2 = store.get(namespace, key)
    print(item2)

    time.sleep(2)

    key2 = "user-2"
    value2 = {
        "name": "Mask"
    }
    store.put(namespace, key2, value2)
    item2 = store.get(namespace, key2)
    print(item2)

Item(namespace=['user', 'info'], key='user-1', value={'name': 'Peter'}, created_at='2026-07-24T07:14:37.709773+00:00', updated_at='2026-07-24T07:22:01.215145+00:00')
Item(namespace=['user', 'info'], key='user-1', value={'name': 'Ruby'}, created_at='2026-07-24T07:14:37.709773+00:00', updated_at='2026-07-24T07:22:02.223936+00:00')
Item(namespace=['user', 'info'], key='user-2', value={'name': 'Mask'}, created_at='2026-07-24T07:22:04.230234+00:00', updated_at='2026-07-24T07:22:04.230234+00:00')


## search()-搜索
- namespace_prefix: 命名空间前缀，类型tuple
- query:语义检索时用于查询的自然语言
- filter：过滤条件，匹配value中的键值对
- limit: 返回的条数限制
- offset: 返回之前跳过的条目数
- refresh_ttl: 刷新时间

### 精准匹配

In [3]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

#### namespace_prefix：命名空间前缀搜索

In [4]:
for item in store.search(("users",)):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-24T07:41:33.676309+00:00', updated_at='2026-07-24T07:41:33.676311+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-07-24T07:41:33.676362+00:00', updated_at='2026-07-24T07:41:33.676363+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-24T07:41:33.676402+00:00', updated_at='2026-07-24T07:41:33.676402+00:00', score=None)


#### filter搜索

In [5]:
resp = store.search(
    ("users",),
    filter={
       "course": "数字电路与模拟电路",
    }
)
for item in resp:
    print(item)

Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-07-24T07:41:33.676362+00:00', updated_at='2026-07-24T07:41:33.676363+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-24T07:41:33.676402+00:00', updated_at='2026-07-24T07:41:33.676402+00:00', score=None)


### 语义搜索
#### 自定义嵌入函数

In [7]:


# 自定义嵌入函数
def embed(text: list[str]) -> list[list[float]]:
    return [[1.0] * 6 for _ in range(len(text))]


index_config = {
    "embed": embed, # 使用嵌入函数或嵌入模型赋值
    "dims": 6,
    "fields": ["$", "course"]
}
store = InMemoryStore(
    index = index_config
)

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)


#### 查看嵌入向量

In [8]:
from rich import print as rprint
rprint(store._vectors)

defaultdict(<function InMemoryStore.__init__.<locals>.<lambda> at 0x000001769FB27420>, {
    ('users', 'Alice', 'memories'): defaultdict(<class 'dict'>, {
        'preferences': {'$': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 'course': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}
    }),
    ('users', 'Bob', 'memories'): defaultdict(<class 'dict'>, {
        'preferences': {'$': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 'course': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}
    }),
    ('users', 'Black', 'memories'): defaultdict(<class 'dict'>, {
        'preferences': {'$': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 'course': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}
    })
})

### 使用向量模型

In [1]:
from langgraph.store.memory import InMemoryStore
from common import init_dashscope_embedding_model

embedding_model = init_dashscope_embedding_model()

index_config = {
    "embed": embedding_model,
    "dims": 1024,
    "fields": ["$"]
}
store = InMemoryStore(
    index = index_config
)

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

#### 语义检索

In [3]:
from rich import print as rprint

result = store.search(("users",), query="模电")

for item in result:
    rprint(item)

result = store.search(("haha",), query="模电")

for item in result:
    rprint(item)

Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': 
'跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-07-24T08:35:18.648500+00:00', 
updated_at='2026-07-24T08:35:18.648502+00:00', score=0.5580789203712088)

Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': 
'羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-24T08:35:18.847140+00:00', 
updated_at='2026-07-24T08:35:18.847142+00:00', score=0.547070420086642)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': 
'跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-24T08:35:18.413822+00:00', 
updated_at='2026-07-24T08:35:18.413825+00:00', score=0.39442567533871314)